# Calibrating and Cataloguing Green-band

In [1]:
# Import various libraries 

import numpy as np
import astropy
import photutils
import ccdproc
from ccdproc import CCDData, combiner
from astropy import units as u
import astropy.io.fits as fits
from astropy.io import ascii
import time
from astropy.visualization import SqrtStretch
from astropy.visualization.mpl_normalize import ImageNormalize
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from photutils.centroids import centroid_com, centroid_1dg, centroid_2dg
from photutils.aperture import CircularAperture
from photutils.aperture import aperture_photometry
from photutils.background import Background2D
from photutils.background import MedianBackground
from photutils.detection import DAOStarFinder
from photutils.segmentation  import detect_sources, deblend_sources, SourceCatalog
from scipy.ndimage import shift
import gc                               

from astropy.coordinates import SkyCoord
from astroquery.gaia import Gaia # this library is used to query astronomical databases 
from photutils.segmentation import SourceCatalog, SegmentationImage


In preparation for Gaia DR4, the Gaia archive is in evolution. Unfortunately, it may be unstable at times and particular types of queries may time out. Please consider registering for a user account (https://www.cosmos.esa.int/web/gaia-users/register). For questions or advice, please contact the Gaia helpdesk (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk).


In [2]:
g_combined = CCDData.read("g_NGC_2547_cropped.fits", unit="adu")
print("Loaded g_combined successfully")

INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
Loaded g_combined successfully


In [3]:
from astroquery.astrometry_net import AstrometryNet
from astropy.io import fits

ast = AstrometryNet()
ast.api_key = 'rahhejaruebqmxrg'  # Get from nova.astrometry.net

# Submit image for plate solving
wcs_header = ast.solve_from_image('g_NGC_2547_cropped.fits', force_image_upload=True)

# Add WCS to original FITS file
with fits.open('g_NGC_2547_cropped.fits', mode='update') as hdul:
    hdul[0].header.update(wcs_header)
    hdul.flush()

print("WCS added successfully!")

Solving..................................................................................................

TimeoutError: ('Solve timed out without success or failure', 15078745)

In [4]:
scim=CCDData.read("g_NGC_2547_cropped.fits", unit="adu")   # Read the image

w = scim.wcs                                      # Read the World Coordinate System
print(w)

med=np.median(scim.data)                          # Median pixel value
scim.data=scim.data-med                           # Use the median for the sky subtraction
mean, median, std = astropy.stats.sigma_clipped_stats(scim.data, sigma=3.0, maxiters=5)    
print('Image stats (mean, median and standard deviation):', mean,median,std)

segimage=detect_sources(scim.data, 2.00*std, 9, connectivity=4, mask=None)

INFO: using the unit adu passed to the FITS reader instead of the unit adu in the FITS file. [astropy.nddata.ccddata]
WCS Keywords

Number of WCS axes: 2
CTYPE : 'RA---TAN-SIP' 'DEC--TAN-SIP'
CUNIT : 'deg' 'deg'
CRVAL : 122.353311568 -48.8902271055
CRPIX : 587.693748474 -116.964008331
PC1_1 PC1_2  : -0.000595667320789 0.000280896353715
PC2_1 PC2_2  : -0.000281051929191 -0.000595612192856
CDELT : 1.0 1.0
NAXIS : 1080  700
Image stats (mean, median and standard deviation): -0.3006382 -0.74451816 9.809436


c:\Users\chris\AppData\Local\Programs\Python\Python312\Lib\site-packages\astropy\wcs\wcs.py:3236: RuntimeWarning: cdelt will be ignored since cd is present
  description.append(s.format(*self.wcs.cdelt))


In [5]:
outfile="segimage.fits"                # Set the output file name
hdu=fits.PrimaryHDU(segimage)          # Define a FITS header for this data
# Convert WCS to header format
wcsheader = w.to_header(relax=True)
hdu.header.extend(wcsheader)
hdu.writeto(outfile, overwrite=True)   # Write the output and overwrite existing files if needed. 


In [6]:
# ── Load image and build source catalogue ─────────────────────────────────────
with fits.open('g_NGC_2547_cropped.fits') as hdul:
    sci_data   = hdul[0].data.astype(float).copy()
    sci_header = hdul[0].header

with fits.open('segimage.fits') as hdul:
    seg_data = hdul[0].data.copy()

w = WCS(sci_header)
sci_data -= np.median(sci_data)
_, _, std = astropy.stats.sigma_clipped_stats(sci_data, sigma=3.0, maxiters=5)

segmap       = SegmentationImage(seg_data)
source_table = SourceCatalog(sci_data, segmap, wcs=w)

# ── Aperture photometry on all detected sources ───────────────────────────────
positions = []
for obj in source_table:
    positions.append((obj.centroid[0], obj.centroid[1]))

apertures  = CircularAperture(positions, r=8.3)
phot_table = aperture_photometry(sci_data, apertures)

print("\nFirst 10 rows of photometry table:")
print(phot_table[:10])
print("\nColumn names:", phot_table.colnames)

# ── Gaia query ────────────────────────────────────────────────────────────────
ra_field  = 122.353125759
dec_field = -48.8904062838

coord  = SkyCoord(ra=ra_field, dec=dec_field, unit='deg', frame='icrs')
radius = u.Quantity(0.25, u.deg)

Gaia.ROW_LIMIT = 5000
print("\nQuerying Gaia DR3...")
j = Gaia.cone_search(coordinate=coord, radius=radius)


r = j.get_results()

mask = (r['phot_g_mean_mag']  > 0) & (r['phot_g_mean_mag']  < 30) & \
       (r['phot_bp_mean_mag'] > 0) & (r['phot_bp_mean_mag'] < 30) & \
       (r['phot_rp_mean_mag'] > 0) & (r['phot_rp_mean_mag'] < 30)
r = r[mask]
r.sort('phot_g_mean_mag')

nrows, ncols = sci_data.shape
gaia_coords  = SkyCoord(ra=r['ra'], dec=r['dec'], unit='deg', frame='icrs')
px, py       = w.world_to_pixel(gaia_coords)

in_image = (px > 20) & (px < ncols-20) & (py > 20) & (py < nrows-20)
r  = r[in_image]
px = px[in_image]
py = py[in_image]
print(f"Gaia sources inside image footprint: {len(r)}")

# ── Match each Gaia star to nearest source in phot_table ─────────────────────
phot_x = np.array(phot_table['xcenter'])
phot_y = np.array(phot_table['ycenter'])

print(" Reference stars (G < 12)")
print(f"{'#':<4} {'RA':>12} {'Dec':>12} {'G_BP':>6} {'G':>6} {'G_RP':>6}  "
      f"{'x_pix':>7} {'y_pix':>7}  {'dist':>5} {'aperture_sum':>14}")
print("-" * 90)
count = 0
for i in range(len(r)):
    if r['phot_g_mean_mag'][i] < 12:
        dists      = np.sqrt((phot_x - px[i])**2 + (phot_y - py[i])**2)
        nearest    = int(np.argmin(dists))
        flux       = phot_table['aperture_sum'][nearest]
        print(f"{count:<4} {r['ra'][i]:>12.5f} {r['dec'][i]:>12.5f} "
              f"{r['phot_bp_mean_mag'][i]:>6.3f} {r['phot_g_mean_mag'][i]:>6.3f} "
              f"{r['phot_rp_mean_mag'][i]:>6.3f}  {px[i]:>7.1f} {py[i]:>7.1f}  "
              f"{dists[nearest]:>5.1f} {flux:>14.1f}")
        count += 1
        if count >= 60:   # added
            break

print("Faint stars (15 < G < 18)")  # was 14 < G < 17
print(f"{'#':<4} {'RA':>12} {'Dec':>12} {'G_BP':>6} {'G':>6} {'G_RP':>6}  "
      f"{'x_pix':>7} {'y_pix':>7}  {'dist':>5} {'aperture_sum':>14}")
print("-" * 90)
count = 0
for i in range(len(r)):
    g = r['phot_g_mean_mag'][i]
    if 15 < g < 18:                     # was 14 < g < 17
        dists   = np.sqrt((phot_x - px[i])**2 + (phot_y - py[i])**2)
        nearest = int(np.argmin(dists))
        flux    = phot_table['aperture_sum'][nearest]
        print(f"{count:<4} {r['ra'][i]:>12.5f} {r['dec'][i]:>12.5f} "
              f"{r['phot_bp_mean_mag'][i]:>6.3f} {g:>6.3f} "
              f"{r['phot_rp_mean_mag'][i]:>6.3f}  {px[i]:>7.1f} {py[i]:>7.1f}  "
              f"{dists[nearest]:>5.1f} {flux:>14.1f}")
        count += 1
        if count >= 60:
            break


First 10 rows of photometry table:
 id      xcenter            ycenter          aperture_sum   
--- ------------------ ------------------ ------------------
  1   4.64617561752514 1.5091268182817483  345.0957251750567
  2  56.88300974947605 2.2189458611291806  4795.365447059685
  3 229.19481832282943 0.9529506424099197   169440.813929697
  4  261.3100606511762 0.6315179270844277  4734.502394661835
  5   425.095855213448  0.561906259688695  2939.444905586718
  6 443.39624819529496  0.677956090894653 2410.4967497698676
  7 487.32728613368545 0.2918785339030813  531.9347517929734
  8    791.92325760133 1.7424104788450492  4765.361263993465
  9  407.2343628876796  5.591985298989658 3473.5056764081755
 10 149.37311268079512  5.398407662343293 1492.9651834483248

Column names: ['id', 'xcenter', 'ycenter', 'aperture_sum']

Querying Gaia DR3...


HTTPError: Error 408: 
Job timeout/aborted.


In [ ]:
# Gaia DR3 → Johnson V transformation
# Using Gaia DR3 documentation (Riello et al. 2021)
# V = G + a0 + a1*(GBP-GRP) + a2*(GBP-GRP)^2 + a3*(GBP-GRP)^3
# Valid for GBP-GRP in range 0.0 to 2.5       


a0 = -0.02704
a1 =  0.01424
a2 = -0.2156
a3 =  0.01426

def gaia_to_V(G, G_BP, G_RP):
    colour = G_BP - G_RP
    V = G + a0 + a1*colour + a2*colour**2 + a3*colour**3
    return V

# ── Reference star ────────────────────────────────────────────────────────────
G_BP_ref = 10.787
G_ref    = 10.702
G_RP_ref = 10.568
F_ref    = 92857.5

# ── Faint star ────────────────────────────────────────────────────────────────
  
G_BP_faint = 15.824
G_faint    = 15.063
G_RP_faint = 14.214 
F_faint    = 1679.1   

V_ref        = gaia_to_V(G_ref,   G_BP_ref,   G_RP_ref)
V_faint_gaia = gaia_to_V(G_faint, G_BP_faint, G_RP_faint)

print("Reference Star")
print(f"  GBP-GRP colour:         {G_BP_ref - G_RP_ref:.3f}")
print(f"  Calibrated V magnitude: {V_ref:.3f}")
print(f"  Measured flux (F_ref):  {F_ref:.1f} ADU")

print("\n Faint Star")
print(f"  GBP-GRP colour:              {G_BP_faint - G_RP_faint:.3f}")
print(f"  Gaia-calibrated V magnitude: {V_faint_gaia:.3f}")
print(f"  Measured flux:               {F_faint:.1f} ADU")

# ── Sanity check ──────────────────────────────────────────────────────────────
V_faint_measured = V_ref - 2.5 * np.log10(F_faint / F_ref)

print("\n Sanity Check")
print(f"  V (from image photometry): {V_faint_measured:.3f}")
print(f"  V (from Gaia DR3 formula): {V_faint_gaia:.3f}")
print(f"  Difference:                {abs(V_faint_measured - V_faint_gaia):.3f} mag")
if abs(V_faint_measured - V_faint_gaia) < 0.5:
    print("  Good agreement")
else:
    print("  Large discrepancy")

In [ ]:
from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry
import numpy as np

# ── Helper functions ──────────────────────────────────────────────────────────
def net_flux(data, pos, r_ap=8.3, r_in=25, r_out=40):
    
    aperture    = CircularAperture([pos], r=r_ap)
    annulus     = CircularAnnulus([pos], r_in=r_in, r_out=r_out)
    phot        = aperture_photometry(data, aperture)
    bkg_phot    = aperture_photometry(data, annulus)
    bkg_per_pix = bkg_phot['aperture_sum'][0] / annulus.area
    flux_net    = phot['aperture_sum'][0] - bkg_per_pix * aperture.area
    return flux_net, bkg_per_pix

def gaia_to_V(G, G_BP, G_RP):
    
    a0, a1, a2, a3 = -0.02704, 0.01424, -0.2156, 0.01426
    return G + a0 + a1*(G_BP-G_RP) + a2*(G_BP-G_RP)**2 + a3*(G_BP-G_RP)**3

def BV_from_gaia(G_BP, G_RP):
    
    return 1.35 * (G_BP - G_RP) - 0.10

def V_to_seestar_green(V, G_BP, G_RP):
    
    BV = BV_from_gaia(G_BP, G_RP)
    return V - 0.07 * BV + 0.01

def seestar_green_to_V(green, G_BP, G_RP):
    
    BV = BV_from_gaia(G_BP, G_RP)
    return green + 0.07 * BV - 0.01

# Reference star
G_BP_ref, G_ref, G_RP_ref = 10.787, 10.702, 10.568
x_ref, y_ref               = 560.6, 90.4

V_ref_johnson  = gaia_to_V(G_ref, G_BP_ref, G_RP_ref)
green_ref      = V_to_seestar_green(V_ref_johnson, G_BP_ref, G_RP_ref)
F_ref, bkg_ref = net_flux(sci_data, (x_ref, y_ref))

print("Reference Star")
print(f"  Johnson V:          {V_ref_johnson:.3f}")
print(f"  SeeStar Green:      {green_ref:.3f}")
print(f"  Net flux:           {F_ref:.1f} ADU")
print(f"  Background/pixel:   {bkg_ref:.3f} ADU")

# Calibrate all sources
maglist   = []   # Calibrated V magnitudes
greenlist = []   # SeeStar Green magnitudes (before V correction)

for obj in phot_table:
    flux_raw = obj['aperture_sum']
    if flux_raw > 0 and F_ref > 0:
        green_mag = green_ref - 2.5 * np.log10(flux_raw / F_ref)
        greenlist.append(green_mag)
        # For V conversion we don't have Gaia colours for every source,
        # so assume a typical cluster star colour (GBP-GRP ~ 0.7, BV ~ 0.85)
        # This is a reasonable approximation for NGC 2547 FGK members
        green_to_V_offset = 0.07 * (1.35 * 0.7 - 0.10) - 0.01   # = +0.047
        maglist.append(green_mag + green_to_V_offset)
    else:
        greenlist.append(np.nan)
        maglist.append(np.nan)

maglist   = np.array(maglist)
greenlist = np.array(greenlist)

# Summary statistics
valid = maglist[np.isfinite(maglist) & (maglist > 0) & (maglist < 25)]
print(f"\n=== Calibration Summary ===")
print(f"  Total sources:         {len(maglist)}")
print(f"  Valid V magnitudes:    {len(valid)}")
print(f"  Magnitude range:       {valid.min():.2f} to {valid.max():.2f}")

# Print first 20 results 
print(f"\n{'idx':>5} {'x':>8} {'y':>8} {'raw_flux':>12} {'Green':>8} {'V_mag':>8}")
print("-" * 55)
for i in range(min(20, len(phot_table))):
    print(f"{i:>5} {phot_table['xcenter'][i].value:>8.2f} "
          f"{phot_table['ycenter'][i].value:>8.2f} "
          f"{phot_table['aperture_sum'][i]:>12.1f} "
          f"{greenlist[i]:>8.3f} {maglist[i]:>8.3f}")

# Histogram of calibrated V magnitudes
import matplotlib.pyplot as plt

bin_edges = np.arange(np.nanmin(valid), np.nanmax(valid) + 0.5, 0.5)

plt.figure(figsize=(9, 5))
plt.hist(valid, bins=bin_edges, edgecolor='black', color='steelblue')
plt.xlabel('V-band Magnitude')
plt.ylabel('Number of Stars')
plt.title('NGC 2547 — Calibrated V-band Magnitude Distribution')
plt.gca().invert_xaxis()
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig("calibrated_V_band_magnitude.png", dpi=300, bbox_inches="tight")
plt.show()



In [ ]:
import pandas as pd
import numpy as np

# Create a dataframe from the results
df = pd.DataFrame({
    "x": phot_table['xcenter'].value,
    "y": phot_table['ycenter'].value,
    "raw_flux": phot_table['aperture_sum'],
    "green_mag": greenlist,
    "V_mag": maglist
})

# Optional: remove NaN magnitudes
df = df.replace([np.inf, -np.inf], np.nan)

# Save to CSV
df.to_csv("maglist_V_catalog.csv", index=False)

print("CSV saved as maglist_V_catalog.csv")

# Calibrating and Cataloguing Blue band  

In [ ]:
b_combined = CCDData.read("b_NGC_2547_cropped.fits", unit="adu")
print("Loaded b_combined successfully")

In [ ]:
from astroquery.astrometry_net import AstrometryNet
from astropy.io import fits

ast = AstrometryNet()
ast.api_key = 'rahhejaruebqmxrg'  # Get from nova.astrometry.net

# Submit image for plate solving
wcs_header = ast.solve_from_image('b_NGC_2547_cropped.fits', force_image_upload=True)

# Add WCS to original FITS file
with fits.open('b_NGC_2547_cropped.fits', mode='update') as hdul:
    hdul[0].header.update(wcs_header)
    hdul.flush()

print("WCS added successfully!")

In [ ]:
scim=CCDData.read("g_NGC_2547_cropped.fits", unit="adu")   # Read the image

w = scim.wcs                                      # Read the World Coordinate System
print(w)

med=np.median(scim.data)                          # Median pixel value
scim.data=scim.data-med                           # Use the median for the sky subtraction
mean, median, std = astropy.stats.sigma_clipped_stats(scim.data, sigma=3.0, maxiters=5)    
print('Image stats (mean, median and standard deviation):', mean,median,std)

segimage=detect_sources(scim.data, 2.00*std, 9, connectivity=4, mask=None)

In [ ]:
# ── Load B-band image and build source catalogue ──────────────────────────────
with fits.open('b_NGC_2547_cropped.fits') as hdul:    # ✏️ B-band filename
    sci_data   = hdul[0].data.astype(float).copy()
    sci_header = hdul[0].header

with fits.open('segimage.fits') as hdul:
    seg_data = hdul[0].data.copy()

w = WCS(sci_header)
sci_data -= np.median(sci_data)
_, _, std = astropy.stats.sigma_clipped_stats(sci_data, sigma=3.0, maxiters=5)

segmap       = SegmentationImage(seg_data)
source_table = SourceCatalog(sci_data, segmap, wcs=w)

# ── Aperture photometry on all detected sources ───────────────────────────────
positions = []
valid_sources = []   # keep track of which sources had valid centroids

for obj in source_table:
    x, y = obj.centroid[0], obj.centroid[1]
    if np.isfinite(x) and np.isfinite(y):   # skip NaN centroids
        positions.append((x, y))
        valid_sources.append(obj)

print(f"Total sources in segmap:       {len(source_table)}")
print(f"Sources with valid centroids:  {len(positions)}")

apertures  = CircularAperture(positions, r=8.3)
phot_table = aperture_photometry(sci_data, apertures)

print("\nFirst 10 rows of photometry table:")
print(phot_table[:10])
print("\nColumn names:", phot_table.colnames)

# ── Gaia query ────────────────────────────────────────────────────────────────
ra_field  = 122.353125759
dec_field = -48.8904062838

coord  = SkyCoord(ra=ra_field, dec=dec_field, unit='deg', frame='icrs')
radius = u.Quantity(0.25, u.deg)

Gaia.ROW_LIMIT = 5000
print("\nQuerying Gaia DR3...")

j = Gaia.cone_search(coordinate=coord, radius=radius)
r = j.get_results()

mask = (r['phot_g_mean_mag']  > 0) & (r['phot_g_mean_mag']  < 30) & \
       (r['phot_bp_mean_mag'] > 0) & (r['phot_bp_mean_mag'] < 30) & \
       (r['phot_rp_mean_mag'] > 0) & (r['phot_rp_mean_mag'] < 30)
r = r[mask]
r.sort('phot_g_mean_mag')

nrows, ncols = sci_data.shape
gaia_coords  = SkyCoord(ra=r['ra'], dec=r['dec'], unit='deg', frame='icrs')
px, py       = w.world_to_pixel(gaia_coords)

in_image = (px > 20) & (px < ncols-20) & (py > 20) & (py < nrows-20)
r  = r[in_image]
px = px[in_image]
py = py[in_image]
print(f"Gaia sources inside image footprint: {len(r)}")

# ── Match each Gaia star to nearest source in phot_table ─────────────────────
phot_x = np.array(phot_table['xcenter'])
phot_y = np.array(phot_table['ycenter'])

print("\n=== Candidate REFERENCE stars (G < 12) ===")
print(f"{'#':<4} {'RA':>12} {'Dec':>12} {'G_BP':>6} {'G':>6} {'G_RP':>6}  "
      f"{'x_pix':>7} {'y_pix':>7}  {'dist':>5} {'aperture_sum':>14}")
print("-" * 90)
count = 0
for i in range(len(r)):
    if r['phot_g_mean_mag'][i] < 12:
        dists   = np.sqrt((phot_x - px[i])**2 + (phot_y - py[i])**2)
        nearest = int(np.argmin(dists))
        flux    = phot_table['aperture_sum'][nearest]
        print(f"{count:<4} {r['ra'][i]:>12.5f} {r['dec'][i]:>12.5f} "
              f"{r['phot_bp_mean_mag'][i]:>6.3f} {r['phot_g_mean_mag'][i]:>6.3f} "
              f"{r['phot_rp_mean_mag'][i]:>6.3f}  {px[i]:>7.1f} {py[i]:>7.1f}  "
              f"{dists[nearest]:>5.1f} {flux:>14.1f}")
        count += 1
        if count >= 60:
            break

print("\n=== Candidate FAINT stars (15 < G < 18) ===")
print(f"{'#':<4} {'RA':>12} {'Dec':>12} {'G_BP':>6} {'G':>6} {'G_RP':>6}  "
      f"{'x_pix':>7} {'y_pix':>7}  {'dist':>5} {'aperture_sum':>14}")
print("-" * 90)
count = 0
for i in range(len(r)):
    g = r['phot_g_mean_mag'][i]
    if 15 < g < 18:
        dists   = np.sqrt((phot_x - px[i])**2 + (phot_y - py[i])**2)
        nearest = int(np.argmin(dists))
        flux    = phot_table['aperture_sum'][nearest]
        print(f"{count:<4} {r['ra'][i]:>12.5f} {r['dec'][i]:>12.5f} "
              f"{r['phot_bp_mean_mag'][i]:>6.3f} {g:>6.3f} "
              f"{r['phot_rp_mean_mag'][i]:>6.3f}  {px[i]:>7.1f} {py[i]:>7.1f}  "
              f"{dists[nearest]:>5.1f} {flux:>14.1f}")
        count += 1
        if count >= 60:
            break

In [ ]:

a0 =  0.00786
a1 =  0.92062
a2 = -0.25444
a3 =  0.05068

def gaia_to_B(G, G_BP, G_RP):
    colour = G_BP - G_RP
    return G + a0 + a1*colour + a2*colour**2 + a3*colour**3

# Reference star 
G_BP_ref = 10.787
G_ref    = 10.702
G_RP_ref = 10.568
F_ref    = 78999.7

# Faint star
  
G_BP_faint = 15.824
G_faint    = 15.063
G_RP_faint = 14.214 
F_faint    = 694.5      

B_ref        = gaia_to_B(G_ref,   G_BP_ref,   G_RP_ref)
B_faint_gaia = gaia_to_B(G_faint, G_BP_faint, G_RP_faint)

print("Reference Star")
print(f"  GBP-GRP colour:         {G_BP_ref - G_RP_ref:.3f}")
print(f"  Calibrated B magnitude: {B_ref:.3f}")
print(f"  Measured flux (F_ref):  {F_ref:.1f} ADU")

print("\n Faint Star")
print(f"  GBP-GRP colour:              {G_BP_faint - G_RP_faint:.3f}")
print(f"  Gaia-calibrated B magnitude: {B_faint_gaia:.3f}")
print(f"  Measured flux:               {F_faint:.1f} ADU")

# Sanity check 
B_faint_measured = B_ref - 2.5 * np.log10(F_faint / F_ref)

print("\n=== Sanity Check ===")
print(f"  B (from image photometry): {B_faint_measured:.3f}")
print(f"  B (from Gaia DR3 formula): {B_faint_gaia:.3f}")
print(f"  Difference:                {abs(B_faint_measured - B_faint_gaia):.3f} mag")
if abs(B_faint_measured - B_faint_gaia) < 0.5:
    print("   Good agreement")
else:
    print("   Large discrepancy")

In [ ]:

# ── Re-query Gaia for the B-band image footprint ─────────────────────────────
# Use the WCS from your B-band image (scim.wcs)
coord = SkyCoord(ra=122.353, dec=-48.890, unit='deg')
radius = u.Quantity(0.35, u.deg)

Gaia.ROW_LIMIT = -1
job     = Gaia.cone_search(coord, radius)
gaia_tb = job.get_results()

print(f"Gaia sources found: {len(gaia_tb)}")

# ── Convert Gaia sky coords to pixel positions using your WCS ─────────────────
gaia_ra  = np.array(gaia_tb['ra'])
gaia_dec = np.array(gaia_tb['dec'])
gaia_bp  = np.array(gaia_tb['phot_bp_mean_mag'])
gaia_g   = np.array(gaia_tb['phot_g_mean_mag'])
gaia_rp  = np.array(gaia_tb['phot_rp_mean_mag'])

# Convert RA/Dec to pixel x/y using the B-band WCS
sky_coords  = SkyCoord(ra=gaia_ra, dec=gaia_dec, unit='deg')
x_pix, y_pix = scim.wcs.all_world2pix(gaia_ra, gaia_dec, 0)

# ── Build gaia_df ─────────────────────────────────────────────────────────────
gaia_df = pd.DataFrame({
    'ra':    gaia_ra,
    'dec':   gaia_dec,
    'G_BP':  gaia_bp,
    'G':     gaia_g,
    'G_RP':  gaia_rp,
    'x_pix': x_pix,
    'y_pix': y_pix
})

# Keep only sources with valid magnitudes and inside the image
gaia_df = gaia_df.dropna(subset=['G_BP', 'G', 'G_RP'])
gaia_df = gaia_df[
    (gaia_df['x_pix'] >= 0) & (gaia_df['x_pix'] <= 1080) &
    (gaia_df['y_pix'] >= 0) & (gaia_df['y_pix'] <= 700)
].reset_index(drop=True)

print(f"Gaia sources inside image with valid mags: {len(gaia_df)}")
print(gaia_df.head())

In [ ]:
from scipy.spatial import cKDTree

# ── Helper functions ──────────────────────────────────────────────────────────
def net_flux(data, pos, r_ap=8.3, r_in=25, r_out=40):
    aperture    = CircularAperture([pos], r=r_ap)
    annulus     = CircularAnnulus([pos], r_in=r_in, r_out=r_out)
    phot        = aperture_photometry(data, aperture)
    bkg_phot    = aperture_photometry(data, annulus)
    bkg_per_pix = bkg_phot['aperture_sum'][0] / annulus.area
    flux_net    = phot['aperture_sum'][0] - bkg_per_pix * aperture.area
    return flux_net, bkg_per_pix

def gaia_to_B(G, G_BP, G_RP):
    a0, a1, a2, a3 = 0.00786, 0.92062, -0.25444, 0.05068
    colour = G_BP - G_RP
    return G + a0 + a1*colour + a2*colour**2 + a3*colour**3

def gaia_to_V(G, G_BP, G_RP):
    """Gaia DR3 → Johnson V transformation"""
    colour = G_BP - G_RP
    return G - 0.01760 - 0.006860*colour - 0.1732*colour**2

# ── Reference star ────────────────────────────────────────────────────────────
G_BP_ref, G_ref, G_RP_ref = 10.787, 10.702, 10.568
x_ref, y_ref               = 560.3, 90.5

B_ref_johnson  = gaia_to_B(G_ref, G_BP_ref, G_RP_ref)
V_ref_johnson  = gaia_to_V(G_ref, G_BP_ref, G_RP_ref)
BV_ref         = B_ref_johnson - V_ref_johnson
blue_ref       = B_ref_johnson - 0.63 * BV_ref + 0.01   # SeeStar Blue of ref star

F_ref, bkg_ref = net_flux(sci_data, (x_ref, y_ref))

print(f"=== Reference Star ===")
print(f"  Johnson B:    {B_ref_johnson:.3f}")
print(f"  Johnson V:    {V_ref_johnson:.3f}")
print(f"  B-V:          {BV_ref:.3f}")
print(f"  SeeStar Blue: {blue_ref:.3f}")
print(f"  Net flux:     {F_ref:.1f} ADU")

# ── Build KD-tree from your existing Gaia pixel positions ─────────────────────
# These come from the Gaia cross-match you already did earlier in the notebook
# gaia_df should have columns: x_pix, y_pix, G_BP, G, G_RP
gaia_xy    = np.column_stack([gaia_df['x_pix'], gaia_df['y_pix']])
gaia_tree  = cKDTree(gaia_xy)

# ── Calibrate all sources ─────────────────────────────────────────────────────
maglist  = []
bluelist = []

for obj in phot_table:
    flux_raw = obj['aperture_sum']
    x        = obj['xcenter'].value
    y        = obj['ycenter'].value

    if flux_raw > 0 and F_ref > 0:
        # Instrument magnitude relative to reference
        blue_mag = blue_ref - 2.5 * np.log10(flux_raw / F_ref)
        bluelist.append(blue_mag)

        # Find nearest Gaia source within 5 pixels
        dist, idx = gaia_tree.query([x, y], k=1)

        if dist < 5.0:
            # Use this star's actual Gaia colour
            G_BP = gaia_df['G_BP'].iloc[idx]
            G_RP = gaia_df['G_RP'].iloc[idx]
            G    = gaia_df['G'].iloc[idx]
            V    = gaia_to_V(G, G_BP, G_RP)

            # Iteratively solve: B = blue_mag + 0.63*(B-V) - 0.01
            B = blue_mag + 0.63 * (1.35*(G_BP-G_RP) - 0.10) - 0.01  # initial guess
            for _ in range(20):
                BV    = B - V
                B_new = blue_mag + 0.63 * BV - 0.01
                if abs(B_new - B) < 1e-5:
                    break
                B = B_new
        else:
            # No Gaia match — fallback to solar colour B-V = 0.65
            V = blue_mag + 0.01 - 0.63 * 0.65   # rough V estimate
            B = blue_mag + 0.63 * 0.65 - 0.01

        maglist.append(B)
    else:
        bluelist.append(np.nan)
        maglist.append(np.nan)

maglist  = np.array(maglist)
bluelist = np.array(bluelist)

valid = maglist[np.isfinite(maglist) & (maglist > 0) & (maglist < 25)]
print(f"\n=== Calibration Summary ===")
print(f"  Total sources:      {len(maglist)}")
print(f"  Valid B magnitudes: {len(valid)}")
print(f"  Magnitude range:    {valid.min():.2f} to {valid.max():.2f}")

In [ ]:
import pandas as pd
import numpy as np

df_b = pd.DataFrame({
    "x":        [obj['xcenter'].value for obj in phot_table],
    "y":        [obj['ycenter'].value for obj in phot_table],
    "raw_flux": [obj['aperture_sum']  for obj in phot_table],
    "blue_mag": bluelist,
    "B_mag":    maglist
})

df_b = df_b.replace([np.inf, -np.inf], np.nan)
df_b.to_csv("maglist_B_catalog.csv", index=False)

valid = maglist[np.isfinite(maglist) & (maglist > 0) & (maglist < 25)]
print(f"=== Saved maglist_B_catalog.csv ===")
print(f"  Total rows:         {len(df_b)}")
print(f"  Valid B magnitudes: {len(valid)}")
print(f"  Magnitude range:    {valid.min():.2f} to {valid.max():.2f}")

# Calibrating and Cataloguing Red band  

In [ ]:
r_combined = CCDData.read("r_NGC_2547_cropped.fits", unit="adu")
print("Loaded r_combined successfully")

In [ ]:
from astroquery.astrometry_net import AstrometryNet
from astropy.io import fits

ast = AstrometryNet()
ast.api_key = 'rahhejaruebqmxrg'  # Get from nova.astrometry.net

# Submit image for plate solving
wcs_header = ast.solve_from_image('r_NGC_2547_cropped.fits', force_image_upload=True)

# Add WCS to original FITS file
with fits.open('r_NGC_2547_cropped.fits', mode='update') as hdul:
    hdul[0].header.update(wcs_header)
    hdul.flush()

print("WCS added successfully!")

In [ ]:
scim=CCDData.read("r_NGC_2547_cropped.fits", unit="adu")   # Read the image

w = scim.wcs                                      # Read the World Coordinate System
print(w)

med=np.median(scim.data)                          # Median pixel value
scim.data=scim.data-med                           # Use the median for the sky subtraction
mean, median, std = astropy.stats.sigma_clipped_stats(scim.data, sigma=3.0, maxiters=5)    
print('Image stats (mean, median and standard deviation):', mean,median,std)

segimage=detect_sources(scim.data, 2.00*std, 9, connectivity=4, mask=None)

In [ ]:
from astroquery.gaia import Gaia
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.wcs import WCS
from astropy.io import fits
from photutils.aperture import CircularAperture, aperture_photometry
from photutils.segmentation import SourceCatalog, SegmentationImage
import astropy.stats
import numpy as np

# ── Load R-band image and build source catalogue ──────────────────────────────
with fits.open('r_NGC_2547_cropped.fits') as hdul:    
    sci_data   = hdul[0].data.astype(float).copy()
    sci_header = hdul[0].header

with fits.open('segimage.fits') as hdul:
    seg_data = hdul[0].data.copy()

w = WCS(sci_header)
sci_data -= np.median(sci_data)
_, _, std = astropy.stats.sigma_clipped_stats(sci_data, sigma=3.0, maxiters=5)

segmap       = SegmentationImage(seg_data)
source_table = SourceCatalog(sci_data, segmap, wcs=w)

# Aperture photometry on all detected sources 
positions = []
valid_sources = []   # keep track of which sources had valid centroids

for obj in source_table:
    x, y = obj.centroid[0], obj.centroid[1]
    if np.isfinite(x) and np.isfinite(y):   # skip NaN centroids
        positions.append((x, y))
        valid_sources.append(obj)

print(f"Total sources in segmap:       {len(source_table)}")
print(f"Sources with valid centroids:  {len(positions)}")

apertures  = CircularAperture(positions, r=8.3)
phot_table = aperture_photometry(sci_data, apertures)

print("\nFirst 10 rows of photometry table:")
print(phot_table[:10])
print("\nColumn names:", phot_table.colnames)

#  Gaia query 
ra_field  = 122.353125759
dec_field = -48.8904062838

coord  = SkyCoord(ra=ra_field, dec=dec_field, unit='deg', frame='icrs')
radius = u.Quantity(0.25, u.deg)

Gaia.ROW_LIMIT = 5000
print("\nQuerying Gaia DR3...")
j = Gaia.cone_search(coordinate=coord, radius=radius)
time.sleep(3)
r = j.get_results()

mask = (r['phot_g_mean_mag']  > 0) & (r['phot_g_mean_mag']  < 30) & \
       (r['phot_bp_mean_mag'] > 0) & (r['phot_bp_mean_mag'] < 30) & \
       (r['phot_rp_mean_mag'] > 0) & (r['phot_rp_mean_mag'] < 30)
r = r[mask]
r.sort('phot_g_mean_mag')

nrows, ncols = sci_data.shape
gaia_coords  = SkyCoord(ra=r['ra'], dec=r['dec'], unit='deg', frame='icrs')
px, py       = w.world_to_pixel(gaia_coords)

in_image = (px > 20) & (px < ncols-20) & (py > 20) & (py < nrows-20)
r  = r[in_image]
px = px[in_image]
py = py[in_image]
print(f"Gaia sources inside image footprint: {len(r)}")

# Matching each Gaia star to nearest source in phot_table 
phot_x = np.array(phot_table['xcenter'])
phot_y = np.array(phot_table['ycenter'])

print("\n=== Candidate REFERENCE stars (G < 12) ===")
print(f"{'#':<4} {'RA':>12} {'Dec':>12} {'G_BP':>6} {'G':>6} {'G_RP':>6}  "
      f"{'x_pix':>7} {'y_pix':>7}  {'dist':>5} {'aperture_sum':>14}")
print("-" * 90)
count = 0
for i in range(len(r)):
    if r['phot_g_mean_mag'][i] < 12:
        dists   = np.sqrt((phot_x - px[i])**2 + (phot_y - py[i])**2)
        nearest = int(np.argmin(dists))
        flux    = phot_table['aperture_sum'][nearest]
        print(f"{count:<4} {r['ra'][i]:>12.5f} {r['dec'][i]:>12.5f} "
              f"{r['phot_bp_mean_mag'][i]:>6.3f} {r['phot_g_mean_mag'][i]:>6.3f} "
              f"{r['phot_rp_mean_mag'][i]:>6.3f}  {px[i]:>7.1f} {py[i]:>7.1f}  "
              f"{dists[nearest]:>5.1f} {flux:>14.1f}")
        count += 1
        if count >= 60:
            break

print("\n=== Candidate FAINT stars (15 < G < 18) ===")
print(f"{'#':<4} {'RA':>12} {'Dec':>12} {'G_BP':>6} {'G':>6} {'G_RP':>6}  "
      f"{'x_pix':>7} {'y_pix':>7}  {'dist':>5} {'aperture_sum':>14}")
print("-" * 90)
count = 0
for i in range(len(r)):
    g = r['phot_g_mean_mag'][i]
    if 15 < g < 18:
        dists   = np.sqrt((phot_x - px[i])**2 + (phot_y - py[i])**2)
        nearest = int(np.argmin(dists))
        flux    = phot_table['aperture_sum'][nearest]
        print(f"{count:<4} {r['ra'][i]:>12.5f} {r['dec'][i]:>12.5f} "
              f"{r['phot_bp_mean_mag'][i]:>6.3f} {g:>6.3f} "
              f"{r['phot_rp_mean_mag'][i]:>6.3f}  {px[i]:>7.1f} {py[i]:>7.1f}  "
              f"{dists[nearest]:>5.1f} {flux:>14.1f}")
        count += 1
        if count >= 60:
            break

In [ ]:

a0 = -0.12879
a1 =  0.24662
a2 = -0.027464
a3 = -0.049465

def gaia_to_R(G, G_BP, G_RP):
    colour = G_BP - G_RP
    return G + a0 + a1*colour + a2*colour**2 + a3*colour**3

# Reference star
G_BP_ref = 10.787
G_ref    = 10.702
G_RP_ref = 10.568
F_ref    = 54261.0

# Faint star 
  
G_BP_faint = 15.824
G_faint    = 15.063
G_RP_faint = 14.214 
F_faint    = 1390.2

R_ref        = gaia_to_R(G_ref,   G_BP_ref,   G_RP_ref)
R_faint_gaia = gaia_to_R(G_faint, G_BP_faint, G_RP_faint)

print("=== Reference Star ===")
print(f"  GBP-GRP colour:         {G_BP_ref - G_RP_ref:.3f}")
print(f"  Calibrated R magnitude: {R_ref:.3f}")
print(f"  Measured flux (F_ref):  {F_ref:.1f} ADU")

print("\n=== Faint Star ===")
print(f"  GBP-GRP colour:              {G_BP_faint - G_RP_faint:.3f}")
print(f"  Gaia-calibrated R magnitude: {R_faint_gaia:.3f}")
print(f"  Measured flux:               {F_faint:.1f} ADU")

# ── Sanity check ──────────────────────────────────────────────────────────────
R_faint_measured = R_ref - 2.5 * np.log10(F_faint / F_ref)

print("\n=== Sanity Check ===")
print(f"  R (from image photometry): {R_faint_measured:.3f}")
print(f"  R (from Gaia DR3 formula): {R_faint_gaia:.3f}")
print(f"  Difference:                {abs(R_faint_measured - R_faint_gaia):.3f} mag")
if abs(R_faint_measured - R_faint_gaia) < 0.5:
    print("  Good agreement")
else:
    print("  Large discrepancy")

In [ ]:
from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry
import numpy as np

# Helper functions 
def net_flux(data, pos, r_ap=8.3, r_in=25, r_out=40):
    aperture    = CircularAperture([pos], r=r_ap)
    annulus     = CircularAnnulus([pos], r_in=r_in, r_out=r_out)
    phot        = aperture_photometry(data, aperture)
    bkg_phot    = aperture_photometry(data, annulus)
    bkg_per_pix = bkg_phot['aperture_sum'][0] / annulus.area
    flux_net    = phot['aperture_sum'][0] - bkg_per_pix * aperture.area
    return flux_net, bkg_per_pix

def gaia_to_R(G, G_BP, G_RP):
    """Gaia DR3 → Cousins R using official colour transformation.
    R = G + a0 + a1*(GBP-GRP) + a2*(GBP-GRP)^2 + a3*(GBP-GRP)^3
    Valid for GBP-GRP in range 0.0 to 2.5  (Riello et al. 2021)
    """
    a0, a1, a2, a3 = -0.12879, 0.24662, -0.027464, -0.049465
    colour = G_BP - G_RP
    return G + a0 + a1*colour + a2*colour**2 + a3*colour**3

def VR_from_gaia(G_BP, G_RP):
    """Approximate Cousins V-R from Gaia colours."""
    return 0.61 * (G_BP - G_RP) - 0.03

def R_to_seestar_red(R, G_BP, G_RP):
    """Convert Cousins R to expected SeeStar Red magnitude.
    SeeStar Red = R + 0.20*(V-R) + 0.02
    """
    VR = VR_from_gaia(G_BP, G_RP)
    return R + 0.20 * VR + 0.02

def seestar_red_to_R(red_mag, G_BP, G_RP):
    """Convert measured SeeStar Red magnitude back to Cousins R."""
    VR = VR_from_gaia(G_BP, G_RP)
    return red_mag - 0.20 * VR - 0.02

# Reference star
G_BP_ref, G_ref, G_RP_ref = 10.787, 10.702, 10.568
x_ref, y_ref               = 560.3, 90.4

R_ref_cousins  = gaia_to_R(G_ref, G_BP_ref, G_RP_ref)
red_ref        = R_to_seestar_red(R_ref_cousins, G_BP_ref, G_RP_ref)
F_ref, bkg_ref = net_flux(sci_data, (x_ref, y_ref))

print("=== Reference Star ===")
print(f"  Cousins R:          {R_ref_cousins:.3f}")
print(f"  SeeStar Red:        {red_ref:.3f}")
print(f"  Net flux:           {F_ref:.1f} ADU")
print(f"  Background/pixel:   {bkg_ref:.3f} ADU")

# Calibrate all sources
maglist  = []   # Calibrated R magnitudes
redlist  = []   # SeeStar Red magnitudes (before R correction)

for obj in phot_table:
    flux_raw = obj['aperture_sum']
    if flux_raw > 0 and F_ref > 0:
        red_mag = red_ref - 2.5 * np.log10(flux_raw / F_ref)
        redlist.append(red_mag)
        # Assume typical cluster colour GBP-GRP ~ 0.7 → V-R ~ 0.40
        red_to_R_offset = -(0.20 * (0.61 * 0.7 - 0.03) + 0.02)   # = -0.093
        maglist.append(red_mag + red_to_R_offset)
    else:
        redlist.append(np.nan)
        maglist.append(np.nan)

maglist = np.array(maglist)
redlist = np.array(redlist)

# ── Summary ───────────────────────────────────────────────────────────────────
valid = maglist[np.isfinite(maglist) & (maglist > 0) & (maglist < 25)]
print(f"\n=== Calibration Summary ===")
print(f"  Total sources:         {len(maglist)}")
print(f"  Valid R magnitudes:    {len(valid)}")
print(f"  Magnitude range:       {valid.min():.2f} to {valid.max():.2f}")

# Print first 20 results 
print(f"\n{'idx':>5} {'x':>8} {'y':>8} {'raw_flux':>12} {'Red':>8} {'R_mag':>8}")
print("-" * 55)
for i in range(min(20, len(phot_table))):
    print(f"{i:>5} {phot_table['xcenter'][i].value:>8.2f} "
          f"{phot_table['ycenter'][i].value:>8.2f} "
          f"{phot_table['aperture_sum'][i]:>12.1f} "
          f"{redlist[i]:>8.3f} {maglist[i]:>8.3f}")

# Histogram 
import matplotlib.pyplot as plt

bin_edges = np.arange(np.nanmin(valid), np.nanmax(valid) + 0.5, 0.5)

plt.figure(figsize=(9, 5))
plt.hist(valid, bins=bin_edges, edgecolor='black', color='firebrick')
plt.xlabel('R-band Magnitude')
plt.ylabel('Number of Stars')
plt.title('NGC 2547 — Calibrated R-band Magnitude Distribution')
plt.gca().invert_xaxis()
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig("calibrated_R_band_magnitude.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
import numpy as np

df_r = pd.DataFrame({
    "x":       phot_table['xcenter'].value,
    "y":       phot_table['ycenter'].value,
    "raw_flux": phot_table['aperture_sum'],
    "red_mag":  redlist,
    "R_mag":    maglist
})

df_r = df_r.replace([np.inf, -np.inf], np.nan)
df_r.to_csv("maglist_R_catalog.csv", index=False)
print("CSV saved as maglist_R_catalog.csv")